# **Text classification using pretrained word embeddings**




In [31]:
from torchtext.vocab import GloVe, vocab

## **Step 1: Import Model**

In [32]:
glove_6B = GloVe(name="6B")

In [33]:
# build vocabulary
vocab = vocab(glove_6B.stoi,specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

In [34]:
vocab(["<unk>","Hello","hello"])

[0, 0, 13075]

## **Step 2: Tokenization**

In [43]:
from torchtext.data.utils import get_tokenizer
from torchtext.datasets import AG_NEWS
from torchtext.data.functional import to_map_style_dataset  
from torch.utils.data import random_split
from torch.utils.data import DataLoader
import torch
import torch.nn as nn

In [36]:
tokenizer = get_tokenizer('basic_english')

Import Dataset

In [37]:
#initialize the train and test iterators
train_iter, test_iter = AG_NEWS()

# define train and test datasets
train_dataset = to_map_style_dataset(train_iter)
test_dataset = to_map_style_dataset(test_iter)

Split train dataset into valid dataset also
 

In [38]:
num_train = int(len(train_dataset)*0.85) 

#randomly split train and valid datasets
split_train, split_valid = random_split(train_dataset,[num_train,(len(train_dataset)-num_train)])

In [39]:
# define class labels
ag_news_label = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tec"}
num_class = len(set([label for (label,text) in train_iter]))
num_class

4

## **Step 3: Data Loader**

Define collate function

In [40]:
def text_pipeline(x):
    x = x.lower()
    return vocab(tokenizer(x))

def label_pipeline(x):
    return int(x)-1

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

In [42]:
def collate_func(batch):
    
    label_list, text_list, offsets = [],[],[0]
    
    for _label, _text in batch:
        label_list.append(label_pipeline(_label))
        process_text = torch.tensor(text_pipeline(_text),dtype=torch.int64)
        text_list.append(process_text)
        offsets.append(process_text.size(0))
    
    label_list = torch.tensor(label_list,dtype=torch.int64)
    offsets = torch.tensor(offsets).cumsum(dim=0)
    text_list = torch.cat(text_list)
    
    return label_list.to(device), text_list.to(device), offsets.to(device)

#### Define Data Loader

In [44]:
batch_size = 64

In [ ]:
train_datalaoder = DataLoader(
    split_train,shuffle=True, batch_size=batch_size, collate_fn=collate_func
)

valid_dataloader = DataLoader(
    split_valid, shuffle=True, batch_size=batch_size, collate_fn=collate_func
)

test_dataloader = DataLoader(
    test_dataset, shuffle=True, batch_size=batch_size, collate_fn=collate_func
)